# Logit Regularizado Cloud — Entrenamiento en Kaggle Notebooks

**Corre en Kaggle Notebooks** (CPU, ~9h disponibles).

**Dataset requerido:** `mecmt07-features` (subir `features_train.parquet` y `features_test.parquet`).

**Flujo:**
1. Preprocesamiento: SimpleImputer (mediana) + StandardScaler
2. `LogisticRegressionCV` con L1, L2 y ElasticNet (5-fold CV AUC)
3. Selección del mejor modelo
4. Guardar pipeline + metadata → descargar desde Output tab

In [1]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import sklearn

print(f'sklearn version : {sklearn.__version__}')
print('Imports OK')

sklearn version : 1.6.1
Imports OK


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════
SEED    = 42
N_FOLDS = 5

DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

# Matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(SEED)

print('=' * 65)
print('CONFIGURACIÓN')
print('=' * 65)
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  SEED      : {SEED}')
print(f'  CPU       : Logit usa CPU (sklearn saga)')
print('=' * 65)
for f in ['features_train.parquet', 'features_test.parquet']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "NO encontrado"}')

CONFIGURACIÓN
  DATA_DIR  : /kaggle/input/datasets/davidguzzi/mecmt07-features
  MODEL_DIR : /kaggle/working
  N_FOLDS   : 5
  SEED      : 42
  CPU       : Logit usa CPU (sklearn saga)
  features_train.parquet: OK
  features_test.parquet: OK


In [3]:
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Encodear columnas categóricas
cat_cols = [c for c in feature_cols if df[c].dtype == 'object']
if cat_cols:
    print(f'  Columnas categóricas encontradas: {cat_cols}')
    for col in cat_cols:
        cats    = pd.concat([df[col], df_test[col]]).dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df[col]      = df[col].map(mapping)
        df_test[col] = df_test[col].map(mapping)
    print('  Encoding completado ✓')
else:
    print('  Sin columnas categóricas')

X_raw       = df[feature_cols].values
y           = df['TARGET'].values
X_test_raw  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'\nTrain: {X_raw.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test_raw.shape}')

Cargando datos...
  Sin columnas categóricas

Train: (307511, 30)  | Default rate: 8.07%
Test : (48744, 30)


In [4]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc, 4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec, 4), Precision=round(prec, 4), F1=round(f1, 4))

print('Funciones auxiliares definidas.')

Funciones auxiliares definidas.


## 1. Modelos Regularizados — L1, L2, ElasticNet

In [5]:
print('Preprocesando (SimpleImputer + StandardScaler)...')
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_imp    = imputer.fit_transform(X_raw)
X_scaled = scaler.fit_transform(X_imp)

X_test_scaled = scaler.transform(imputer.transform(X_test_raw))

print(f'  X_scaled: {X_scaled.shape} — NaNs: {np.isnan(X_scaled).sum()}')

Cs  = np.logspace(-4, 2, 20)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f'  Grilla de C: {len(Cs)} valores [{Cs.min():.5f} — {Cs.max():.1f}]')
print(f'  CV folds   : {N_FOLDS}')

Preprocesando (SimpleImputer + StandardScaler)...
  X_scaled: (307511, 30) — NaNs: 0
  Grilla de C: 20 valores [0.00010 — 100.0]
  CV folds   : 5


In [6]:
# ── L1 (Lasso) ─────────────────────────────────────────────────────────────────
print('Ajustando Logit L1 (Lasso) con LogisticRegressionCV...')
t0 = time.time()
lr_l1 = LogisticRegressionCV(
    Cs=Cs, cv=skf, penalty='l1', solver='saga',
    scoring='roc_auc', class_weight='balanced',
    max_iter=2000, n_jobs=-1, random_state=SEED
)
lr_l1.fit(X_scaled, y)

best_C_l1   = lr_l1.C_[0]
cv_aucs_l1  = lr_l1.scores_[1].mean(axis=0)
best_auc_l1 = cv_aucs_l1.max()
n_nonzero_l1 = int((lr_l1.coef_[0] != 0).sum())

print(f'  Completado en {time.time()-t0:.0f}s')
print(f'  Mejor C (L1): {best_C_l1:.6f} | CV AUC: {best_auc_l1:.5f}')
print(f'  Coeficientes != 0: {n_nonzero_l1} / {len(feature_cols)}')

Ajustando Logit L1 (Lasso) con LogisticRegressionCV...
  Completado en 95s
  Mejor C (L1): 23.357215 | CV AUC: 0.74976
  Coeficientes != 0: 30 / 30


In [7]:
# ── L2 (Ridge) ─────────────────────────────────────────────────────────────────
print('Ajustando Logit L2 (Ridge) con LogisticRegressionCV...')
t0 = time.time()
lr_l2 = LogisticRegressionCV(
    Cs=Cs, cv=skf, penalty='l2', solver='saga',
    scoring='roc_auc', class_weight='balanced',
    max_iter=2000, n_jobs=-1, random_state=SEED
)
lr_l2.fit(X_scaled, y)

best_C_l2   = lr_l2.C_[0]
cv_aucs_l2  = lr_l2.scores_[1].mean(axis=0)
best_auc_l2 = cv_aucs_l2.max()

print(f'  Completado en {time.time()-t0:.0f}s')
print(f'  Mejor C (L2): {best_C_l2:.6f} | CV AUC: {best_auc_l2:.5f}')

Ajustando Logit L2 (Ridge) con LogisticRegressionCV...
  Completado en 59s
  Mejor C (L2): 0.001833 | CV AUC: 0.74976


In [8]:
# ── ElasticNet ─────────────────────────────────────────────────────────────────
l1_ratios  = [0.1, 0.3, 0.5, 0.7, 0.9]
en_results = []

print('Ajustando ElasticNet (grid l1_ratio × C)...')
for ratio in l1_ratios:
    t0 = time.time()
    lr_en = LogisticRegressionCV(
        Cs=Cs, cv=skf, penalty='elasticnet', l1_ratios=[ratio],
        solver='saga', scoring='roc_auc', class_weight='balanced',
        max_iter=2000, n_jobs=-1, random_state=SEED
    )
    lr_en.fit(X_scaled, y)
    auc_cv = lr_en.scores_[1].mean(axis=0).max()
    best_c = lr_en.C_[0]
    print(f'  l1_ratio={ratio:.1f} | best C={best_c:.5f} | CV AUC={auc_cv:.5f}  ({time.time()-t0:.0f}s)')
    en_results.append({'l1_ratio': ratio, 'best_C': best_c, 'cv_auc': auc_cv,
                       'model': lr_en})

best_en    = max(en_results, key=lambda x: x['cv_auc'])
lr_best_en = best_en['model']
print(f'\nMejor ElasticNet: l1_ratio={best_en["l1_ratio"]} | CV AUC={best_en["cv_auc"]:.5f}')

Ajustando ElasticNet (grid l1_ratio × C)...
  l1_ratio=0.1 | best C=0.14384 | CV AUC=0.74976  (82s)
  l1_ratio=0.3 | best C=48.32930 | CV AUC=0.74976  (94s)
  l1_ratio=0.5 | best C=23.35721 | CV AUC=0.74976  (97s)
  l1_ratio=0.7 | best C=23.35721 | CV AUC=0.74976  (93s)
  l1_ratio=0.9 | best C=23.35721 | CV AUC=0.74976  (98s)

Mejor ElasticNet: l1_ratio=0.1 | CV AUC=0.74976


In [9]:
# ── Selección del mejor modelo ─────────────────────────────────────────────────
model_map = {
    'L1 (Lasso)'  : (lr_l1, best_auc_l1, best_C_l1),
    'L2 (Ridge)'  : (lr_l2, best_auc_l2, best_C_l2),
    f'ElasticNet (l1={best_en["l1_ratio"]})': (lr_best_en, best_en['cv_auc'], best_en['best_C'])
}

print('\nComparación de modelos:')
for name, (mdl, auc, C) in model_map.items():
    print(f'  {name:<30}: CV AUC = {auc:.5f}  |  best C = {C:.6f}')

best_name, (best_lr, best_auc, best_C) = max(
    model_map.items(), key=lambda x: x[1][1]
)
print(f'\n  ▶ Ganador: {best_name}  (CV AUC = {best_auc:.5f})')

y_prob_train = best_lr.predict_proba(X_scaled)[:, 1]
train_auc    = roc_auc_score(y, y_prob_train)
print(f'  Train AUC: {train_auc:.5f}')


Comparación de modelos:
  L1 (Lasso)                    : CV AUC = 0.74976  |  best C = 23.357215
  L2 (Ridge)                    : CV AUC = 0.74976  |  best C = 0.001833
  ElasticNet (l1=0.1)           : CV AUC = 0.74976  |  best C = 0.143845

  ▶ Ganador: L2 (Ridge)  (CV AUC = 0.74976)
  Train AUC: 0.74999


## 2. Metricas sobre Train Completo

In [10]:
metrics_train = compute_metrics(y, y_prob_train, label=f'Logit {best_name} (train in-sample)')
for m in [metrics_train]:
    print(f"\n{m['Model']}")
    print(f"  AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Precision={m['Precision']:.4f}  F1={m['F1']:.4f}")
    print(f"  TP={m['TP']:,}  TN={m['TN']:,}  FP={m['FP']:,}  FN={m['FN']:,}")
print(f'\n  CV AUC ({N_FOLDS}-fold, LogisticRegressionCV): {best_auc:.5f}')
display(pd.DataFrame([metrics_train]).set_index('Model'))


Logit L2 (Ridge) (train in-sample)
  AUC=0.7500  Recall=0.6736  Precision=0.1629  F1=0.2623
  TP=16,723  TN=196,730  FP=85,956  FN=8,102

  CV AUC (5-fold, LogisticRegressionCV): 0.74976


,AUC,N,P,TP,TN,FP,FN,Recall,Precision,F1
Model,,,,,,,,,,
Logit L2 (Ridge) (train in-sample),0.75,307511,102679,16723,196730,85956,8102,0.6736,0.1629,0.2623


In [11]:
cv_aucs_en = lr_best_en.scores_[1].mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
for aucs, color, marker, label_str in [
        (cv_aucs_l1, '#e74c3c', 'o', 'L1 (Lasso)'),
        (cv_aucs_l2, '#3498db', 's', 'L2 (Ridge)'),
        (cv_aucs_en, '#27ae60', '^', f'ElasticNet (l1={best_en["l1_ratio"]})')]:
    ax.semilogx(Cs, aucs, color=color, lw=2, marker=marker, ms=4, label=label_str)
for C_val, color in [(best_C_l1, '#e74c3c'), (best_C_l2, '#3498db'), (best_en['best_C'], '#27ae60')]:
    ax.axvline(C_val, color=color, linestyle='--', lw=1, alpha=0.7)
ax.set_xlabel('C (regularizacion inversa)')
ax.set_ylabel(f'CV AUC ({N_FOLDS}-fold)')
ax.set_title('CV AUC vs C — L1, L2, ElasticNet (semilog)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_reg_cv_curves.png', dpi=120)
plt.show()
print('Grafico guardado: logit_reg_cv_curves.png')

Grafico guardado: logit_reg_cv_curves.png


In [12]:
fig, ax = plt.subplots(figsize=(7, 6))
colors_roc = ['#e74c3c', '#3498db', '#27ae60']
for (name, (mdl, auc_cv, _)), color in zip(model_map.items(), colors_roc):
    y_prob_m  = mdl.predict_proba(X_scaled)[:, 1]
    auc_train = roc_auc_score(y, y_prob_m)
    fpr, tpr, _ = roc_curve(y, y_prob_m)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}\n(train={auc_train:.4f}, cv={auc_cv:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR (1 - Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title('ROC Comparativa — L1 vs L2 vs ElasticNet')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_reg_roc_curve.png', dpi=120)
plt.show()
print('Grafico guardado: logit_reg_roc_curve.png')

Grafico guardado: logit_reg_roc_curve.png


## 3. Coeficientes del Mejor Modelo

In [13]:
coef_s = pd.Series(best_lr.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
top20  = coef_s.head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coeficiente (escala estandarizada)')
ax.set_title(f'Top-20 Coeficientes — {best_name}\n(rojo=positivo, azul=negativo)')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_reg_coeficientes.png', dpi=120)
plt.show()
print('Grafico guardado: logit_reg_coeficientes.png')

Grafico guardado: logit_reg_coeficientes.png


In [14]:
fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde   = gaussian_kde(probs, bw_method=0.1)
    xs    = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title(f'Distribucion de scores — {best_name}')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_reg_score_distribution.png', dpi=120)
plt.show()
print('Grafico guardado: logit_reg_score_distribution.png')

Grafico guardado: logit_reg_score_distribution.png


## 4. Guardar Modelo y Metadata

In [15]:
model_path = MODEL_DIR / 'logit_reg_cloud_best.pkl'
meta_path  = MODEL_DIR / 'logit_reg_cloud_metadata.json'

joblib.dump({
    'model'       : best_lr,
    'imputer'     : imputer,
    'scaler'      : scaler,
    'feature_cols': feature_cols,
    'model_name'  : best_name
}, model_path)

metadata = {
    'model_type'     : 'logit_regularized',
    'best_model_name': best_name,
    'best_C'         : float(best_C),
    'best_cv_auc'    : round(best_auc, 6),
    'train_auc'      : round(train_auc, 6),
    'n_folds'        : N_FOLDS,
    'all_results'    : [
        {'model': k, 'cv_auc': round(v[1], 6), 'best_C': round(float(v[2]), 8)}
        for k, v in model_map.items()
    ],
    'feature_cols'   : feature_cols,
    'sklearn_version': sklearn.__version__,
    'timestamp'      : pd.Timestamp.now().isoformat()
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
print(f'  {model_path.name:<45} ({model_path.stat().st_size/1e6:.2f} MB)')
print(f'  {meta_path.name}')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
print(f'    - {model_path.name}')
print(f'    - {meta_path.name}')
print('\n>>> Luego correr localmente: logit_cloud_predict.ipynb')

ARTEFACTOS GUARDADOS
  logit_reg_cloud_best.pkl                      (0.03 MB)
  logit_reg_cloud_metadata.json

>>> Descargar desde el Output tab de Kaggle:
    - logit_reg_cloud_best.pkl
    - logit_reg_cloud_metadata.json

>>> Luego correr localmente: logit_cloud_predict.ipynb


## Resumen Final — Logit Regularizado Cloud

In [16]:
import platform
print('=' * 65)
print('LOGIT REGULARIZADO CLOUD — RESUMEN FINAL')
print('=' * 65)
print(f'  Mejor modelo         : {best_name}')
print(f'  Mejor C              : {best_C:.6f}')
print(f'  CV AUC (5-fold)      : {best_auc:.5f}')
print(f'  Train AUC (in-sample): {train_auc:.5f}')
print(f'  Gap                  : {abs(train_auc - best_auc):.5f}')
print(f'  n_train              : {X_scaled.shape[0]:,}')
print('─' * 65)
print('  Comparacion L1 vs L2 vs EN:')
for k, v in model_map.items():
    print(f'    {k:<35}: CV AUC={v[1]:.5f}')
print('─' * 65)
print(f'  sklearn   : {sklearn.__version__}')
print(f'  numpy     : {np.__version__}')
print(f'  Python    : {platform.python_version()}')
print(f'  Timestamp : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)

LOGIT REGULARIZADO CLOUD — RESUMEN FINAL
  Mejor modelo         : L2 (Ridge)
  Mejor C              : 0.001833
  CV AUC (5-fold)      : 0.74976
  Train AUC (in-sample): 0.74999
  Gap                  : 0.00023
  n_train              : 307,511
─────────────────────────────────────────────────────────────────
  Comparacion L1 vs L2 vs EN:
    L1 (Lasso)                         : CV AUC=0.74976
    L2 (Ridge)                         : CV AUC=0.74976
    ElasticNet (l1=0.1)                : CV AUC=0.74976
─────────────────────────────────────────────────────────────────
  sklearn   : 1.6.1
  numpy     : 2.0.2
  Python    : 3.12.12
  Timestamp : 2026-02-25 18:06:22
